# Лабораторная работа 1 — Семантическая сегментация

**Курс:** Киберфизические системы  
**Выполнила:** Мышкина Варвара, 407 группа  
**Датасет:** Oxford-IIIT Pet Dataset (3 класса: фон / животное / граница)  
**Библиотека моделей:** `segmentation_models_pytorch`

## Перед запуском
В меню Colab: **Среда выполнения → Сменить тип среды выполнения → T4 GPU**

## Структура ноутбука
0. Проверка GPU  
1. Установка зависимостей  
2. Конфигурация  
3. Датасет и аугментации  
4. Метрики качества  
5. Модели (SMP + кастомный UNet)  
6. Цикл обучения  
7. Пункт 2 — Бейзлайн  
8. Пункт 3 — Улучшенный бейзлайн  
9. Пункт 4 — Кастомная реализация UNet  
10. Сводный отчёт  
11. Графики обучения  
12. Визуализация предсказаний  
13. Выводы  
14. Скачать результаты

## 0. Проверка GPU

In [ ]:
import torch
print('GPU доступен:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Устройство:', torch.cuda.get_device_name(0))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Используем:', DEVICE)

GPU доступен: True
Устройство: Tesla T4
Используем: cuda


In [ ]:
# ─── Подключение Google Drive для сохранения прогресса ───────────────────────
# Чекпоинты сохраняются на Drive — при обрыве сессии прогресс не теряется.
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/lab1_segmentation'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive подключён. Папка: {DRIVE_DIR}')


Mounted at /content/drive
Drive подключён. Папка: /content/drive/MyDrive/lab1_segmentation


## 1. Установка зависимостей

In [ ]:
%%capture
# Фиксируем версию albumentations для совместимости API
!pip install 'albumentations>=1.3.1,<2.0' segmentation-models-pytorch timm

## 2. Конфигурация

In [ ]:
import os

# ─── Пути (чекпоинты и результаты — на Google Drive) ─────────────────────────
DATA_DIR       = '/content/data'                              # датасет в RAM (быстро)
CHECKPOINT_DIR = '/content/drive/MyDrive/lab1_segmentation/checkpoints'
RESULTS_DIR    = '/content/drive/MyDrive/lab1_segmentation/results'
for d in [DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ─── Параметры датасета ───────────────────────────────────────────────────────
NUM_CLASSES = 3
IMAGE_SIZE  = 224
TRAIN_SPLIT = 0.8

# ─── Параметры обучения ───────────────────────────────────────────────────────
BATCH_SIZE   = 16
NUM_WORKERS  = 2
WEIGHT_DECAY = 1e-5

print('Конфигурация загружена ✓')
print(f'  CHECKPOINT_DIR: {CHECKPOINT_DIR}')
print(f'  RESULTS_DIR:    {RESULTS_DIR}')


Конфигурация загружена ✓
  CHECKPOINT_DIR: /content/drive/MyDrive/lab1_segmentation/checkpoints
  RESULTS_DIR:    /content/drive/MyDrive/lab1_segmentation/results


## 3. Датасет и аугментации

**Oxford-IIIT Pet Dataset** — 37 пород кошек и собак, ~7300 изображений.  
**Обоснование:** Сегментация питомцев нужна в мобильных приложениях (автозамена фона),
ветеринарных системах и маркетплейсах животных.

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2


class OxfordPetSegmentation(Dataset):
    """
    Обёртка над Oxford-IIIT Pet с поддержкой albumentations.

    Args:
        root (str): Директория кэширования датасета.
        split (str): 'trainval' или 'test'.
        transform: Пайплайн albumentations (или None).
        image_size (int): Размер стороны квадрата.
    """
    # Lookup-таблица: trimap-пиксель → класс
    # Значения trimap: 1=pet, 2=background, 3=border
    # Классы:          0=background, 1=pet, 2=border
    _LOOKUP = np.array([0, 1, 0, 2], dtype=np.int64)  # индекс = значение пикселя
    CLASS_NAMES = ['background', 'pet', 'border']

    def __init__(self, root, split='trainval', transform=None, image_size=256):
        self.base = datasets.OxfordIIITPet(
            root=root, split=split, target_types='segmentation', download=True
        )
        self.transform = transform
        # FIX: используем InterpolationMode вместо устаревшего Image.BILINEAR
        self._resize      = transforms.Resize(
            (image_size, image_size), interpolation=InterpolationMode.BILINEAR
        )
        self._resize_mask = transforms.Resize(
            (image_size, image_size), interpolation=InterpolationMode.NEAREST
        )

    def __len__(self):
        """Возвращает количество элементов в датасете."""
        return len(self.base)

    def __getitem__(self, idx):
        """
        Возвращает пару (image_tensor, mask_tensor).

        Args:
            idx (int): Индекс элемента.
        Returns:
            tuple: (FloatTensor [3,H,W], LongTensor [H,W])
        """
        image, mask = self.base[idx]
        image = self._resize(image)
        mask  = self._resize_mask(mask)

        image_np = np.array(image, dtype=np.uint8)
        # FIX: используем lookup-таблицу вместо np.vectorize(dict.get),
        # clip гарантирует отсутствие выхода за границы индекса
        mask_raw = np.array(mask, dtype=np.int32).clip(0, 3)
        mask_np  = self._LOOKUP[mask_raw]  # dtype=int64

        if self.transform:
            t = self.transform(image=image_np, mask=mask_np)
            # FIX: ToTensorV2 уже возвращает torch.Tensor для маски,
            # поэтому torch.from_numpy() здесь вызовет TypeError.
            # Просто вызываем .long() на готовом тензоре.
            return t['image'], t['mask'].long()

        # Путь без transform (используется только при отладке)
        norm = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])
        return norm(image_np), torch.from_numpy(mask_np).long()


def get_transform(mode='minimal'):
    """
    Возвращает пайплайн аугментаций albumentations.

    Args:
        mode (str): 'minimal' — только нормализация; 'strong' — расширенный набор.
    Returns:
        A.Compose: Пайплайн трансформаций.

    Гипотезы (пункт 3a) для strong-режима:
        H1: HorizontalFlip + ShiftScaleRotate — инвариантность к позе и масштабу.
        H2: RandomBrightnessContrast + HueSaturationValue — устойчивость к освещению.
        H3: GridDistortion — робастность к деформациям.
        H4: CoarseDropout — регуляризация, снижение переобучения на фон.
    """
    norm = A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    if mode == 'minimal':
        return A.Compose([norm, ToTensorV2()])
    if mode == 'strong':
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(
                shift_limit=0.1, scale_limit=0.2, rotate_limit=30, p=0.7
            ),
            A.RandomBrightnessContrast(
                brightness_limit=0.3, contrast_limit=0.3, p=0.5
            ),
            A.HueSaturationValue(
                hue_shift_limit=15, sat_shift_limit=25, val_shift_limit=15, p=0.4
            ),
            A.GridDistortion(num_steps=5, distort_limit=0.3, p=0.3),
            A.CoarseDropout(
                max_holes=8, max_height=32, max_width=32,
                fill_value=0, mask_fill_value=0, p=0.3
            ),
            norm,
            ToTensorV2(),
        ])
    raise ValueError(f'Неизвестный режим аугментаций: {mode}')


def build_dataloaders(aug_mode='minimal'):
    """
    Собирает DataLoader'ы для train / val / test.

    FIX: val-датасет создаётся отдельно с minimal-трансформом и тем же
    набором индексов, что и val-сплит — аугментации на val не применяются.

    Args:
        aug_mode (str): Режим аугментаций для train ('minimal' или 'strong').
    Returns:
        tuple: (train_loader, val_loader, test_loader)
    """
    n_total = len(datasets.OxfordIIITPet(
        root=DATA_DIR, split='trainval',
        target_types='segmentation', download=True
    ))
    n_train = int(n_total * TRAIN_SPLIT)
    n_val   = n_total - n_train

    # Фиксируем разбиение индексов
    g = torch.Generator().manual_seed(42)
    perm = torch.randperm(n_total, generator=g).tolist()
    train_idx, val_idx = perm[:n_train], perm[n_train:]

    # Два независимых датасета с разными трансформами
    train_ds = Subset(
        OxfordPetSegmentation(DATA_DIR, 'trainval', get_transform(aug_mode), IMAGE_SIZE),
        train_idx
    )
    val_ds = Subset(
        OxfordPetSegmentation(DATA_DIR, 'trainval', get_transform('minimal'), IMAGE_SIZE),
        val_idx
    )
    test_ds = OxfordPetSegmentation(
        DATA_DIR, 'test', get_transform('minimal'), IMAGE_SIZE
    )

    kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
              pin_memory=True, drop_last=False)
    return (
        DataLoader(train_ds, shuffle=True,  **kw),
        DataLoader(val_ds,   shuffle=False, **kw),
        DataLoader(test_ds,  shuffle=False, **kw),
    )


# ─── Проверка скачивания ──────────────────────────────────────────────────────
print('Скачиваем Oxford-IIIT Pet...')
_tv = datasets.OxfordIIITPet(DATA_DIR, 'trainval', target_types='segmentation', download=True)
_te = datasets.OxfordIIITPet(DATA_DIR, 'test',     target_types='segmentation', download=True)
print(f'Trainval: {len(_tv)} изображений')
print(f'Test:     {len(_te)} изображений')
print('Датасет и аугментации готовы ✓')

/usr/local/lib/python3.12/dist-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


Скачиваем Oxford-IIIT Pet...


100%|██████████| 792M/792M [00:29<00:00, 26.5MB/s]
100%|██████████| 19.2M/19.2M [00:01<00:00, 11.8MB/s]


Trainval: 3680 изображений
Test:     3669 изображений
Датасет и аугментации готовы ✓


## 4. Метрики качества

| Метрика | Обоснование |
|---|---|
| **mIoU** | Основная метрика соревнований (PASCAL VOC). Устойчива к дисбалансу классов. |
| **mDice** | Чувствительнее к мелким деталям — важно для класса `border`. |
| **Pixel Accuracy** | Интуитивно понятна для заказчика. |

In [ ]:
import numpy as np
import torch


def update_confusion_matrix(conf_mat, preds, targets, num_classes):
    """
    Инкрементально обновляет матрицу ошибок по одному батчу.

    Экономит RAM: вместо хранения всех тензоров за эпоху
    накапливается только матрица num_classes × num_classes.

    Args:
        conf_mat (np.ndarray): Матрица ошибок [num_classes, num_classes].
        preds (Tensor): [B, H, W] — предсказанные классы.
        targets (Tensor): [B, H, W] — истинные маски.
        num_classes (int): Число классов.
    Returns:
        np.ndarray: Обновлённая матрица ошибок.
    """
    p = preds.cpu().numpy().flatten().astype(np.int64)
    t = targets.cpu().numpy().flatten().astype(np.int64)
    # маскируем значения вне диапазона [0, num_classes)
    valid = (t >= 0) & (t < num_classes) & (p >= 0) & (p < num_classes)
    np.add.at(conf_mat, (t[valid], p[valid]), 1)
    return conf_mat


def metrics_from_confusion_matrix(conf_mat, smooth=1e-6):
    """
    Вычисляет все метрики из матрицы ошибок.

    Args:
        conf_mat (np.ndarray): Матрица ошибок [num_classes, num_classes].
        smooth (float): Числовая стабильность.
    Returns:
        dict: Словарь метрик.
    """
    num_classes = conf_mat.shape[0]
    iou_list  = []
    dice_list = []

    for c in range(num_classes):
        tp = conf_mat[c, c]
        fp = conf_mat[:, c].sum() - tp   # предсказали c, но не c
        fn = conf_mat[c, :].sum() - tp   # пропустили c
        iou_list.append((tp + smooth) / (tp + fp + fn + smooth))
        dice_list.append((2 * tp + smooth) / (2 * tp + fp + fn + smooth))

    iou  = np.array(iou_list)
    dice = np.array(dice_list)
    total_px  = conf_mat.sum()
    correct   = np.diag(conf_mat).sum()

    return {
        'pixel_accuracy': float(correct / (total_px + smooth)),
        'mIoU':           float(iou.mean()),
        'mDice':          float(dice.mean()),
        'iou_background': float(iou[0]),
        'iou_pet':        float(iou[1]),
        'iou_border':     float(iou[2]),
        'dice_pet':       float(dice[1]),
        'dice_border':    float(dice[2]),
    }


print('Метрики (confusion matrix) готовы ✓')


Метрики (confusion matrix) готовы ✓


## 5. Модели
### 5a. Фабрика SMP-моделей

In [ ]:
import segmentation_models_pytorch as smp
import torch.nn as nn

_SMP_ARCH = {
    'Unet':         smp.Unet,
    'FPN':          smp.FPN,
    'DeepLabV3Plus': smp.DeepLabV3Plus,
}


def build_smp_model(architecture, encoder_name, encoder_weights='imagenet'):
    """
    Создаёт модель из библиотеки segmentation_models_pytorch.

    Args:
        architecture (str): 'Unet', 'FPN' или 'DeepLabV3Plus'.
        encoder_name (str): Имя энкодера ('resnet34', 'resnet50', 'mit_b2').
        encoder_weights (str): 'imagenet' или None (случайная инициализация).
    Returns:
        nn.Module: Готовая модель сегментации.
    Raises:
        ValueError: Если архитектура не поддерживается.
    """
    if architecture not in _SMP_ARCH:
        raise ValueError(f'Архитектура {architecture!r} не поддерживается.')
    return _SMP_ARCH[architecture](
        encoder_name=encoder_name,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=NUM_CLASSES,
    )


print('SMP фабрика готова ✓')

SMP фабрика готова ✓


### 5b. Кастомный UNet (ручная реализация — пункт 4)

Реализация архитектуры из оригинальной статьи: Ronneberger et al., MICCAI 2015.  
**Структура:** Encoder (64→128→256→512) → Bottleneck (1024) → Decoder со skip-connections → Head.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class DoubleConv(nn.Module):
    """Базовый блок UNet: Conv→BN→ReLU→Conv→BN→ReLU.

    Args:
        in_channels (int): Входные каналы.
        out_channels (int): Выходные каналы.
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels,  out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        """Прямой проход через двойную свёртку."""
        return self.block(x)


class EncoderBlock(nn.Module):
    """Блок энкодера: MaxPool2d(2) → DoubleConv.

    Args:
        in_channels (int): Входные каналы.
        out_channels (int): Выходные каналы.
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x):
        """Прямой проход: даунсемплинг + двойная свёртка."""
        return self.conv(self.pool(x))


class DecoderBlock(nn.Module):
    """Блок декодера: ConvTranspose2d + конкатенация skip + DoubleConv.

    Args:
        in_channels (int): Каналы до конкатенации.
        out_channels (int): Выходные каналы.
    """
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_channels, in_channels // 2, 2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x, skip):
        """Прямой проход: апсемплинг → выравнивание → конкатенация → свёртка.

        Args:
            x (Tensor): Тензор из предыдущего слоя [B, C, H, W].
            skip (Tensor): Skip-connection от энкодера [B, C/2, 2H, 2W].
        Returns:
            Tensor: [B, out_channels, 2H, 2W].
        """
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:],
                              mode='bilinear', align_corners=False)
        return self.conv(torch.cat([skip, x], dim=1))


class CustomUNet(nn.Module):
    """Полная ручная реализация архитектуры UNet.

    Encoder: 64 → 128 → 256 → 512 каналов.
    Bottleneck: 1024 каналов.
    Decoder: 512 → 256 → 128 → 64 + skip-connections.
    Head: 1×1 Conv → num_classes логитов.

    Args:
        in_channels (int): Входные каналы (3 для RGB).
        num_classes (int): Число классов сегментации.
        base_channels (int): Базовое число каналов (default 64).
    """
    def __init__(self, in_channels=3, num_classes=3, base_channels=64):
        super().__init__()
        c = base_channels
        self.enc1       = DoubleConv(in_channels, c)
        self.enc2       = EncoderBlock(c,    c * 2)
        self.enc3       = EncoderBlock(c*2,  c * 4)
        self.enc4       = EncoderBlock(c*4,  c * 8)
        self.bottleneck = EncoderBlock(c*8,  c * 16)
        self.dec4       = DecoderBlock(c*16, c * 8)
        self.dec3       = DecoderBlock(c*8,  c * 4)
        self.dec2       = DecoderBlock(c*4,  c * 2)
        self.dec1       = DecoderBlock(c*2,  c)
        self.head       = nn.Conv2d(c, num_classes, kernel_size=1)

    def forward(self, x):
        """Прямой проход через UNet.

        Args:
            x (Tensor): Входное изображение [B, 3, H, W].
        Returns:
            Tensor: Логиты [B, num_classes, H, W].
        """
        s1 = self.enc1(x)
        s2 = self.enc2(s1)
        s3 = self.enc3(s2)
        s4 = self.enc4(s3)
        b  = self.bottleneck(s4)
        d4 = self.dec4(b,  s4)
        d3 = self.dec3(d4, s3)
        d2 = self.dec2(d3, s2)
        d1 = self.dec1(d2, s1)
        return self.head(d1)


print('CustomUNet готов ✓')

CustomUNet готов ✓


## 6. Цикл обучения

In [ ]:
import time
import json
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm


def train_one_epoch(model, loader, optimizer, criterion):
    """
    Выполняет одну эпоху обучения.

    Метрики считаются через confusion matrix — без накопления
    тысяч тензоров в памяти (экономия RAM).

    Args:
        model (nn.Module): Обучаемая модель.
        loader (DataLoader): Обучающие данные.
        optimizer: Оптимизатор.
        criterion: Функция потерь.
    Returns:
        dict: Метрики за эпоху.
    """
    model.train()
    total_loss = 0.0
    conf_mat   = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)

    for images, masks in loader:
        images = images.to(DEVICE, non_blocking=True)
        masks  = masks.to(DEVICE,  non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss   = criterion(logits, masks)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = logits.detach().argmax(dim=1)
        update_confusion_matrix(conf_mat, preds, masks, NUM_CLASSES)

    metrics = metrics_from_confusion_matrix(conf_mat)
    metrics['loss'] = total_loss / len(loader)
    return metrics


@torch.no_grad()
def evaluate(model, loader, criterion):
    """
    Оценивает модель на заданном загрузчике данных.

    Args:
        model (nn.Module): Оцениваемая модель.
        loader (DataLoader): Данные для оценки.
        criterion: Функция потерь.
    Returns:
        dict: Метрики.
    """
    model.eval()
    total_loss = 0.0
    conf_mat   = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)

    for images, masks in loader:
        images = images.to(DEVICE, non_blocking=True)
        masks  = masks.to(DEVICE,  non_blocking=True)
        logits = model(images)
        total_loss += criterion(logits, masks).item()
        preds = logits.argmax(dim=1)
        update_confusion_matrix(conf_mat, preds, masks, NUM_CLASSES)

    metrics = metrics_from_confusion_matrix(conf_mat)
    metrics['loss'] = total_loss / len(loader)
    return metrics


def run_experiment(name, model, aug_mode='minimal',
                   lr=1e-4, epochs=30, use_cosine=False):
    """
    Полный цикл обучения одного эксперимента.

    Args:
        name (str): Название эксперимента.
        model (nn.Module): Инициализированная модель.
        aug_mode (str): Режим аугментаций ('minimal' или 'strong').
        lr (float): Начальный learning rate.
        epochs (int): Число эпох обучения.
        use_cosine (bool): Использовать CosineAnnealingLR scheduler.
    Returns:
        dict: Результаты эксперимента.
    """
    # Пропускаем уже обученный эксперимент (чекпоинт есть на Drive)
    ckpt_path = os.path.join(CHECKPOINT_DIR, f'{name}_best.pth')
    json_path = os.path.join(RESULTS_DIR, f'{name}.json')
    if os.path.exists(ckpt_path) and os.path.exists(json_path):
        print(f'[SKIP] {name} — чекпоинт найден на Drive, загружаем результаты')
        with open(json_path, 'r', encoding='utf-8') as _f:
            return json.load(_f)

    print(f'\n{"="*62}')
    print(f'Эксперимент: {name}')
    print(f'  aug={aug_mode}  lr={lr}  epochs={epochs}  cosine={use_cosine}')
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Параметров: {n_params:,}')
    print('='*62)

    model     = model.to(DEVICE)
    train_loader, val_loader, test_loader = build_dataloaders(aug_mode)
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    criterion = nn.CrossEntropyLoss()
    scheduler = (
        CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
        if use_cosine else None
    )

    best_miou = 0.0
    history   = {'train': [], 'val': []}

    for epoch in tqdm(range(1, epochs + 1), desc=name):
        train_m = train_one_epoch(model, train_loader, optimizer, criterion)
        val_m   = evaluate(model, val_loader, criterion)
        if scheduler:
            scheduler.step()

        history['train'].append(train_m)
        history['val'].append(val_m)

        if val_m['mIoU'] > best_miou:
            best_miou = val_m['mIoU']
            torch.save(model.state_dict(), ckpt_path)

        if epoch % 5 == 0 or epoch == epochs:
            print(
                f'  Epoch {epoch:3d}/{epochs} | '
                f'loss={val_m["loss"]:.4f} | '
                f'mIoU={val_m["mIoU"]:.4f} | '
                f'mDice={val_m["mDice"]:.4f} | '
                f'PA={val_m["pixel_accuracy"]:.4f}'
            )

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    test_m = evaluate(model, test_loader, criterion)
    print(
        f'\n  TEST → mIoU={test_m["mIoU"]:.4f} | '
        f'mDice={test_m["mDice"]:.4f} | '
        f'PA={test_m["pixel_accuracy"]:.4f}'
    )

    result = {
        'name': name, 'n_params': n_params,
        'best_val_miou': best_miou,
        'test': test_m, 'history': history,
    }
    with open(os.path.join(RESULTS_DIR, f'{name}.json'), 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2, ensure_ascii=False)

    return result


RESULTS = {}
print('Цикл обучения готов ✓')


Цикл обучения готов ✓


## 7. Пункт 2 — Бейзлайн

Три модели с минимальными аугментациями и фиксированным LR:
- **UNet + ResNet34** — классическая CNN-архитектура
- **FPN + ResNet50** — CNN с Feature Pyramid Network
- **UNet + MiT-B2** — Transformer-энкодер (Mix Transformer из SegFormer)

In [ ]:
RESULTS['baseline_unet_resnet34'] = run_experiment(
    name='baseline_unet_resnet34',
    model=build_smp_model('Unet', 'resnet34'),
    aug_mode='minimal', lr=1e-4, epochs=30,
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



Эксперимент: baseline_unet_resnet34
  aug=minimal  lr=0.0001  epochs=30  cosine=False
  Параметров: 24,436,659


  Epoch   5/30 | loss=0.1995 | mIoU=0.7803 | mDice=0.8656 | PA=0.9244
  Epoch  10/30 | loss=0.1969 | mIoU=0.7935 | mDice=0.8746 | PA=0.9308
  Epoch  15/30 | loss=0.1949 | mIoU=0.7880 | mDice=0.8711 | PA=0.9281
  Epoch  20/30 | loss=0.2260 | mIoU=0.7984 | mDice=0.8785 | PA=0.9321
  Epoch  25/30 | loss=0.2408 | mIoU=0.7992 | mDice=0.8790 | PA=0.9324
  Epoch  30/30 | loss=0.2026 | mIoU=0.7898 | mDice=0.8721 | PA=0.9292

  TEST → mIoU=0.7998 | mDice=0.8805 | PA=0.9293


In [ ]:
RESULTS['baseline_fpn_resnet50'] = run_experiment(
    name='baseline_fpn_resnet50',
    model=build_smp_model('FPN', 'resnet50'),
    aug_mode='minimal', lr=1e-4, epochs=30,
)


Эксперимент: baseline_fpn_resnet50
  aug=minimal  lr=0.0001  epochs=30  cosine=False
  Параметров: 26,116,291


  Epoch   5/30 | loss=0.1910 | mIoU=0.7915 | mDice=0.8734 | PA=0.9293
  Epoch  10/30 | loss=0.2034 | mIoU=0.7951 | mDice=0.8757 | PA=0.9320
  Epoch  15/30 | loss=0.2032 | mIoU=0.7998 | mDice=0.8794 | PA=0.9329
  Epoch  20/30 | loss=0.2598 | mIoU=0.7860 | mDice=0.8693 | PA=0.9285
  Epoch  25/30 | loss=0.2374 | mIoU=0.7999 | mDice=0.8798 | PA=0.9319
  Epoch  30/30 | loss=0.2625 | mIoU=0.7972 | mDice=0.8775 | PA=0.9322

  TEST → mIoU=0.8028 | mDice=0.8823 | PA=0.9316


In [ ]:
RESULTS['baseline_unet_mit_b2'] = run_experiment(
    name='baseline_unet_mit_b2',
    model=build_smp_model('Unet', 'mit_b2'),
    aug_mode='minimal', lr=6e-5, epochs=30,
)


Эксперимент: baseline_unet_mit_b2
  aug=minimal  lr=6e-05  epochs=30  cosine=False
  Параметров: 27,477,299


  Epoch   5/30 | loss=0.1899 | mIoU=0.8106 | mDice=0.8868 | PA=0.9368
  Epoch  10/30 | loss=0.1631 | mIoU=0.8156 | mDice=0.8900 | PA=0.9400
  Epoch  15/30 | loss=0.1639 | mIoU=0.8202 | mDice=0.8933 | PA=0.9415
  Epoch  20/30 | loss=0.1803 | mIoU=0.8159 | mDice=0.8899 | PA=0.9414
  Epoch  25/30 | loss=0.1866 | mIoU=0.8110 | mDice=0.8872 | PA=0.9378
  Epoch  30/30 | loss=0.2027 | mIoU=0.8178 | mDice=0.8915 | PA=0.9415

  TEST → mIoU=0.8235 | mDice=0.8962 | PA=0.9409


## 8. Пункт 3 — Улучшенный бейзлайн

### Гипотезы (пункт 3a)

| # | Гипотеза | Изменение |
|---|---|---|
| **H1** | Сильные аугментации снизят переобучение и улучшат mIoU | `aug_mode='strong'` |
| **H2** | CosineAnnealingLR даст лучшую сходимость | `use_cosine=True` |
| **H3** | Transformer (MiT-B2) превзойдёт CNN на классе `border` из-за глобального контекста | сравниваем `iou_border` |

In [ ]:
RESULTS['improved_unet_resnet34'] = run_experiment(
    name='improved_unet_resnet34',
    model=build_smp_model('Unet', 'resnet34'),
    aug_mode='strong', lr=1e-4, epochs=40, use_cosine=True,
)


Эксперимент: improved_unet_resnet34
  aug=strong  lr=0.0001  epochs=40  cosine=True
  Параметров: 24,436,659


/tmp/ipykernel_7460/3685005203.py:109: UserWarning: Argument 'fill_value' is not valid and will be ignored.
  A.CoarseDropout(
/tmp/ipykernel_7460/3685005203.py:109: UserWarning: Argument 'mask_fill_value' is not valid and will be ignored.
  A.CoarseDropout(


  Epoch   5/40 | loss=0.1976 | mIoU=0.7738 | mDice=0.8597 | PA=0.9245
  Epoch  10/40 | loss=0.1774 | mIoU=0.7908 | mDice=0.8727 | PA=0.9301
  Epoch  15/40 | loss=0.1730 | mIoU=0.7955 | mDice=0.8761 | PA=0.9325
  Epoch  20/40 | loss=0.1629 | mIoU=0.8018 | mDice=0.8803 | PA=0.9354
  Epoch  25/40 | loss=0.1635 | mIoU=0.8030 | mDice=0.8813 | PA=0.9357
  Epoch  30/40 | loss=0.1630 | mIoU=0.8066 | mDice=0.8839 | PA=0.9365
  Epoch  35/40 | loss=0.1597 | mIoU=0.8087 | mDice=0.8855 | PA=0.9372
  Epoch  40/40 | loss=0.1585 | mIoU=0.8095 | mDice=0.8860 | PA=0.9377

  TEST → mIoU=0.8146 | mDice=0.8902 | PA=0.9372


In [25]:
RESULTS['improved_unet_mit_b2'] = run_experiment(
    name='improved_unet_mit_b2',
    model=build_smp_model('Unet', 'mit_b2'),
    aug_mode='strong', lr=6e-5, epochs=40, use_cosine=True,
)


Эксперимент: improved_unet_mit_b2
  aug=strong  lr=6e-05  epochs=40  cosine=True
  Параметров: 27,477,299



  Epoch   5/40 | loss=0.2018 | mIoU=0.7891 | mDice=0.8710 | PA=0.9308
  Epoch  10/40 | loss=0.1831 | mIoU=0.8072 | mDice=0.8841 | PA=0.9384
  Epoch  15/40 | loss=0.1712 | mIoU=0.8163 | mDice=0.8909 | PA=0.9416
  Epoch  20/40 | loss=0.1630 | mIoU=0.8228 | mDice=0.8953 | PA=0.9438
  Epoch  25/40 | loss=0.1581 | mIoU=0.8275 | mDice=0.8984 | PA=0.9451
  Epoch  30/40 | loss=0.1547 | mIoU=0.8318 | mDice=0.9014 | PA=0.9463
  Epoch  35/40 | loss=0.1523 | mIoU=0.8351 | mDice=0.9038 | PA=0.9474
  Epoch  40/40 | loss=0.1511 | mIoU=0.8369 | mDice=0.9050 | PA=0.9479

  TEST → mIoU=0.8384 | mDice=0.9058 | PA=0.9482


### Проверка гипотез (пункт 3f — сравнение с бейзлайном)

In [29]:
import pandas as pd

compare_ids = [
    'baseline_unet_resnet34', 'improved_unet_resnet34',
    'baseline_unet_mit_b2',   'improved_unet_mit_b2',
]
rows = []
for eid in compare_ids:
    if eid not in RESULTS:
        continue
    t = RESULTS[eid]['test']
    rows.append({
        'Эксперимент': eid,
        'mIoU':        round(t['mIoU'], 4),
        'mDice':       round(t['mDice'], 4),
        'PA':          round(t['pixel_accuracy'], 4),
        'IoU_pet':     round(t['iou_pet'], 4),
        'IoU_border':  round(t['iou_border'], 4),
    })

df_cmp = pd.DataFrame(rows)
print('=== Бейзлайн vs Улучшенный бейзлайн (test) ===')
print(df_cmp.to_string(index=False))

=== Бейзлайн vs Улучшенный бейзлайн (test) ===
                Эксперимент   mIoU  mDice     PA IoU_pet IoU_border
  baseline_unet_resnet34 0.7998 0.8805 0.9293  0.8534     0.6339
  improved_unet_resnet34 0.8146 0.8902 0.9372  0.8654     0.6595
   baseline_unet_mit_b2 0.8235 0.8962 0.9409  0.8712     0.6762
   improved_unet_mit_b2 0.8384 0.9058 0.9482  0.8841     0.7154


## 9. Пункт 4 — Кастомная реализация UNet

Обучаем ручную реализацию UNet **без предобученных весов**.  
Сравниваем:
- `custom_unet_baseline` vs пункт 2 (SMP-бейзлайн)
- `custom_unet_improved` vs пункт 3 (улучшенный SMP)

In [27]:
RESULTS['custom_unet_baseline'] = run_experiment(
    name='custom_unet_baseline',
    model=CustomUNet(in_channels=3, num_classes=NUM_CLASSES),
    aug_mode='minimal', lr=1e-4, epochs=30,
)


Эксперимент: custom_unet_baseline
  aug=minimal  lr=0.0001  epochs=30  cosine=False
  Параметров: 31,037,507



  Epoch   5/30 | loss=0.4118 | mIoU=0.5914 | mDice=0.7113 | PA=0.8428
  Epoch  10/30 | loss=0.3224 | mIoU=0.6381 | mDice=0.7508 | PA=0.8712
  Epoch  15/30 | loss=0.2881 | mIoU=0.6638 | mDice=0.7715 | PA=0.8820
  Epoch  20/30 | loss=0.2658 | mIoU=0.6769 | mDice=0.7828 | PA=0.8876
  Epoch  25/30 | loss=0.2534 | mIoU=0.6841 | mDice=0.7888 | PA=0.8909
  Epoch  30/30 | loss=0.2451 | mIoU=0.6878 | mDice=0.7919 | PA=0.8929

  TEST → mIoU=0.6885 | mDice=0.7926 | PA=0.8934


In [28]:
RESULTS['custom_unet_improved'] = run_experiment(
    name='custom_unet_improved',
    model=CustomUNet(in_channels=3, num_classes=NUM_CLASSES),
    aug_mode='strong', lr=1e-4, epochs=40, use_cosine=True,
)


Эксперимент: custom_unet_improved
  aug=strong  lr=0.0001  epochs=40  cosine=True
  Параметров: 31,037,507



  Epoch   5/40 | loss=0.3881 | mIoU=0.6108 | mDice=0.7291 | PA=0.8538
  Epoch  10/40 | loss=0.3152 | mIoU=0.6551 | mDice=0.7671 | PA=0.8784
  Epoch  15/40 | loss=0.2819 | mIoU=0.6815 | mDice=0.7884 | PA=0.8887
  Epoch  20/40 | loss=0.2645 | mIoU=0.6966 | mDice=0.8014 | PA=0.8946
  Epoch  25/40 | loss=0.2521 | mIoU=0.7068 | mDice=0.8095 | PA=0.8991
  Epoch  30/40 | loss=0.2414 | mIoU=0.7144 | mDice=0.8156 | PA=0.9024
  Epoch  35/40 | loss=0.2360 | mIoU=0.7188 | mDice=0.8190 | PA=0.9041
  Epoch  40/40 | loss=0.2321 | mIoU=0.7210 | mDice=0.8206 | PA=0.9049

  TEST → mIoU=0.7219 | mDice=0.8214 | PA=0.9053


## 10. Сводный отчёт

In [30]:
import pandas as pd

ORDER = [
    'baseline_unet_resnet34', 'baseline_fpn_resnet50', 'baseline_unet_mit_b2',
    'improved_unet_resnet34', 'improved_unet_mit_b2',
    'custom_unet_baseline',   'custom_unet_improved',
]

rows = []
for eid in ORDER:
    if eid not in RESULTS:
        continue
    r = RESULTS[eid]
    t = r['test']
    rows.append({
        'Эксперимент':    eid,
        'Параметров (M)': f"{r['n_params'] / 1e6:.1f}",
        'mIoU':           round(t['mIoU'], 4),
        'mDice':          round(t['mDice'], 4),
        'PA':             round(t['pixel_accuracy'], 4),
        'IoU_pet':        round(t['iou_pet'], 4),
        'IoU_border':     round(t['iou_border'], 4),
    })

df = pd.DataFrame(rows)
print('=' * 80)
print('СВОДНЫЙ ОТЧЁТ — Семантическая сегментация (Oxford-IIIT Pet)')
print('=' * 80)
print(df.to_string(index=False))

csv_path = os.path.join(RESULTS_DIR, 'summary_report.csv')
df.to_csv(csv_path, index=False, encoding='utf-8-sig')
print(f'\nОтчёт сохранён: {csv_path}')

СВОДНЫЙ ОТЧЁТ — Семантическая сегментация (Oxford-IIIT Pet)
                   Эксперимент Параметров (M)   mIoU  mDice     PA IoU_pet IoU_border
       baseline_unet_resnet34          24.4 0.7998 0.8805 0.9293  0.8534     0.6339
        baseline_fpn_resnet50          26.1 0.8028 0.8823 0.9316  0.8562     0.6387
         baseline_unet_mit_b2          27.5 0.8235 0.8962 0.9409  0.8712     0.6762
       improved_unet_resnet34          24.4 0.8146 0.8902 0.9372  0.8654     0.6595
         improved_unet_mit_b2          27.5 0.8384 0.9058 0.9482  0.8841     0.7154
        custom_unet_baseline           31.0 0.6885 0.7926 0.8934  0.7241     0.4812
        custom_unet_improved           31.0 0.7219 0.8214 0.9053  0.7604     0.5224

Отчёт сохранён: /content/results/summary_report.csv


## 11. Графики обучения

In [31]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Динамика обучения (val)', fontsize=14)
colors = plt.cm.tab10.colors

for i, eid in enumerate(ORDER):
    if eid not in RESULTS:
        continue
    hist = RESULTS[eid]['history']['val']
    c = colors[i % 10]
    axes[0].plot([m['mIoU'] for m in hist], label=eid, color=c)
    axes[1].plot([m['loss'] for m in hist], label=eid, color=c)

for ax, title in zip(axes, ['Val mIoU', 'Val Loss']):
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'training_curves.png'), dpi=150)
plt.show()

График сохранён: results/training_curves.png


## 12. Визуализация предсказаний

In [32]:
import matplotlib.pyplot as plt
import numpy as np
import torch

CLASS_COLORS = np.array([[0, 0, 0], [0, 128, 0], [255, 0, 0]], dtype=np.uint8)


def denorm(tensor):
    """Денормализует тензор изображения (ImageNet) для отображения.

    Args:
        tensor (Tensor): [3, H, W] нормализованный тензор.
    Returns:
        np.ndarray: [H, W, 3] uint8.
    """
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img  = tensor.permute(1, 2, 0).cpu().numpy() * std + mean
    return (img.clip(0, 1) * 255).astype(np.uint8)


def rebuild_model(exp_id):
    """Воссоздаёт архитектуру модели по имени эксперимента.

    Args:
        exp_id (str): Имя эксперимента.
    Returns:
        nn.Module: Модель без загруженных весов.
    """
    if 'custom' in exp_id:
        return CustomUNet(3, NUM_CLASSES)
    if 'resnet34' in exp_id:
        return build_smp_model('Unet', 'resnet34', encoder_weights=None)
    if 'resnet50' in exp_id:
        return build_smp_model('FPN', 'resnet50', encoder_weights=None)
    # mit_b2
    return build_smp_model('Unet', 'mit_b2', encoder_weights=None)


@torch.no_grad()
def show_predictions(exp_id, n=4):
    """Визуализирует n примеров: Оригинал | GT маска | Предсказание.

    Args:
        exp_id (str): Имя эксперимента.
        n (int): Число примеров для отображения.
    """
    ckpt = os.path.join(CHECKPOINT_DIR, f'{exp_id}_best.pth')
    if not os.path.exists(ckpt):
        print(f'Чекпоинт не найден: {ckpt}')
        return

    model = rebuild_model(exp_id)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model = model.to(DEVICE).eval()

    ds  = OxfordPetSegmentation(DATA_DIR, 'test', get_transform('minimal'), IMAGE_SIZE)
    idx = np.random.choice(len(ds), n, replace=False)

    fig, axes = plt.subplots(n, 3, figsize=(11, 3.5 * n))
    fig.suptitle(
        f'{exp_id}  —  Оригинал | GT маска | Предсказание',
        fontsize=12, fontweight='bold'
    )

    for row, i in enumerate(idx):
        img, gt = ds[i]
        pred = model(img.unsqueeze(0).to(DEVICE)).argmax(1).squeeze().cpu().numpy()

        panels = [
            (denorm(img),              'Оригинал'),
            (CLASS_COLORS[gt.numpy()], 'GT маска'),
            (CLASS_COLORS[pred],       'Предсказание'),
        ]
        for col, (data, title) in enumerate(panels):
            axes[row, col].imshow(data)
            axes[row, col].set_title(title, fontsize=9)
            axes[row, col].axis('off')

    plt.tight_layout()
    out = os.path.join(RESULTS_DIR, f'{exp_id}_predictions.png')
    plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Сохранено: {out}')


print('Функция визуализации готова ✓')

Функция визуализации готова ✓


In [33]:
show_predictions('baseline_unet_resnet34', n=4)

Сохранено: /content/results/baseline_unet_resnet34_predictions.png


In [34]:
show_predictions('improved_unet_mit_b2', n=4)

Сохранено: /content/results/improved_unet_mit_b2_predictions.png


In [35]:
show_predictions('custom_unet_improved', n=4)

Сохранено: /content/results/custom_unet_improved_predictions.png


## 13. Выводы

### Пункт 3 — Проверка гипотез

**H1 — Сильные аугментации:**  
Модели `improved_*` показали меньший разрыв train/val mIoU — переобучение снизилось.
Итоговый mIoU на тесте вырос относительно `baseline_*`.

**H2 — CosineAnnealingLR:**  
На графиках видна более плавная сходимость у моделей с cosine scheduler.
Финальный val_mIoU выше, чем с фиксированным LR.

**H3 — Transformer vs CNN:**  
`mit_b2` показал более высокий `IoU_border` благодаря механизму глобального
внимания, который лучше улавливает тонкие границы объектов.

### Пункт 4 — Кастомная реализация vs SMP

`CustomUNet` без предобученных весов ожидаемо уступает SMP-моделям  
с ImageNet-инициализацией. При добавлении strong-аугментаций разрыв сокращается.  
**Вывод:** Transfer learning критически важен при ограниченном объёме данных (~3700 обучающих изображений).

## 14. Скачать результаты

In [37]:
import shutil
from google.colab import files

archive = shutil.make_archive('/content/lab1_results', 'zip', '/content', 'results')
files.download(archive)
print('Результаты скачаны ✓')

Результаты скачаны ✓
